In [57]:
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [58]:
## collecting all of the neccesary data
df_Ageo_top20 = pd.read_csv("datasets_rq4/df_Ageo_top20.csv")
df_AID_TOP20_YMA = pd.read_csv("datasets_rq4/df_AID_TOP20_YMA.csv")
df_delay_type = pd.read_csv("datasets_rq4/df_delay_type.csv")

world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")


Merging all datasets to one

In [59]:
# changing month column to numbers for df_AID_TOP20_YMA
df_AID_TOP20_YMA['Month_Lobt_num'] = pd.to_datetime(df_AID_TOP20_YMA['Month_Lobt'], format='%B').dt.month


# merging the year and month column 
df_AID_TOP20_YMA['month_year'] = (
    df_AID_TOP20_YMA['Year_Lobt'].astype(str) + '-' +
    df_AID_TOP20_YMA['Month_Lobt_num'].astype(str).str.zfill(2))

df_delay_type['month_year'] = df_delay_type['Year'].astype(str) + '-' + df_delay_type['Month'].astype(str).str.zfill(2)


# merging df_Ageo and df_AT
df_total0 = pd.merge(df_AID_TOP20_YMA, df_Ageo_top20,
                    left_on='APT_ICAO', right_on='ident', how='left')

# merging df_delay_type
df_total1 = pd.merge(df_total0, df_delay_type,
                     left_on=["APT_ICAO", "month_year"], right_on=["APT_ICAO", "month_year"], how="left")

# Sorting by date
df_total1 = df_total1.sort_values('month_year').reset_index(drop=True)


## Code for the map

In [61]:
# Min and Max delay, so the colorscheme stays consistent during the months
min_delay = df_total1['Total Delay (TD)'].min()
max_delay = df_total1['Total Delay (TD)'].max()

# Code for the map
map_fig = px.scatter_map(
    df_total1,
    lat="latitude_deg",
    lon="longitude_deg",
    size="Total_Flights_Period",
    color="Total Delay (TD)",
    color_continuous_scale=["green", "yellow", "red"],
    range_color=[min_delay, max_delay],
    map_style="carto-positron",
    zoom=4,
    width=1000,
    height=700,
    animation_frame="month_year",
    size_max=40,
    hover_name="name", 
    hover_data={
        "Total Delay (TD)": True,
        "Total Flights (TF)": True,
        "Total_Flights_Period": True,
        "Avg Delay per Movement (in min)": True,
        "Avg Proportion of Delay (%)": True,
        "Delay_Ratio": True,
        "delay_type": True},
)

# Chaning the hover text
hover_text=("<b>%{hovertext}</b><br>" +
    "Total Delay: %{customdata[0]} minutes <br>" +
    "Total Flights: %{customdata[1]}<br>" +
    "Total Flights Period: %{customdata[2]}<br>" +
    "Average delay per Movement: %{customdata[3]:.2f} minutes <br>" +
    "Average Proportion of Delay: %{customdata[4]:.2f} % <br>" +
    "Delay ratio: %{customdata[5]:.2f} % <br>" +
    "Most common cause of delay: %{customdata[6]} <br>")

map_fig.update_traces(hovertemplate=hover_text)

# Ensuring hover_text is applied for all animation frames
for frame in map_fig.frames:
    for trace in frame.data:
        trace.hovertemplate = hover_text

map_fig.show()
